# Multiclass prediction on CPU
Load a complete fine-tuned checkpoint and restore predictions to the original image size.
Set `checkpoint` to the full `birefnet-multiclass-v1` model saved by training.
The label PNG contains class IDs (0 through `num_classes - 1`); its values are not display intensities.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
from PIL import Image
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.build.model import build_predictor
from src.config import load_config
from src.predict.inference import predict
from src.prepare.spec import PreprocessSpec

cfg = load_config(ROOT / 'config/train.yaml')
checkpoint = ROOT / 'run/<run-id>/weights/best_miou.pth'
device = torch.device('cpu')
model = build_predictor(cfg, str(checkpoint), device)
spec = PreprocessSpec.from_meta(model.loaded_meta)
image = np.asarray(Image.open(ROOT / 'asset/sample.webp').convert('RGB'))


In [ ]:
labels = predict(model, image, size=spec.size, mode=spec.mode, tiles=(1,))
assert labels.shape == image.shape[:2]
assert labels.dtype == np.uint8
assert labels.max() < model.num_classes
Image.fromarray(labels).save(ROOT / 'labels.png')

# A palette is only for visualization; labels.png retains the class IDs.
palette = np.array([[0, 0, 0], [230, 85, 85], [80, 190, 120], [80, 130, 230]], dtype=np.uint8)
if model.num_classes > len(palette):
    rng = np.random.default_rng(0)
    palette = rng.integers(0, 256, (model.num_classes, 3), dtype=np.uint8)
display(Image.fromarray(image))
display(Image.fromarray(palette[labels]))


In [ ]:
# Python can return all per-class float32 probabilities (C,H,W).
probs = predict(model, image, output_mode='probability', size=spec.size, mode=spec.mode)
assert probs.shape == (model.num_classes, *image.shape[:2])
np.testing.assert_allclose(probs.sum(axis=0), 1.0, atol=1e-6)

# Selecting one class returns a uint8 probability image, as the API does.
class_id = 3
probability = predict(model, image, output_mode='probability', class_id=class_id,
                      size=spec.size, mode=spec.mode)
display(Image.fromarray(probability))
